<a href="https://colab.research.google.com/github/Supan-Roy/financial-scam-analysis/blob/main/notebooks/08_synthetic_data_augmentation_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook 08: Synthetic Data Augmentation for Binary Financial Scam Detection

## 1. Research Objective

This notebook investigates whether the synthetic financial scam datasets previously developed and used in the multi-level classification experiments can improve binary financial scam detection.

The investigation reuses the established synthetic scam datasets:

- V3.1 synthetic financial scam dataset.
- LLM-authored financial scam dataset.

Both datasets were previously incorporated into the research pipeline for subtype classification, fraudulent intent analysis, social-engineering tactic recognition, and cross-generation evaluation.

This notebook extends their use to binary scam-versus-legitimate classification.

The experiments compare three training configurations using the same DistilBERT architecture:

- **Experiment A — Real-only:** Real financial scam + real legitimate communications.
- **Experiment B — Synthetic:** Previously established synthetic financial scam data + real legitimate communications.
- **Experiment C — Combined:** Real financial scam + previously established synthetic financial scam data + real legitimate communications.

Experiment A has already been completed in Notebook 06. Its trained model and benchmark results will be reused rather than retrained.

## 2. Experimental Sequence

1. **Existing-model evaluation:** Evaluate the saved real-only DistilBERT model on synthetic scam communications paired with real legitimate communications. No additional training or fine-tuning will be performed.
2. **Synthetic-only training:** Train a new model using the established synthetic scam datasets and real legitimate communications.
3. **Combined training:** Train a new model using real scam, synthetic scam, and real legitimate communications.
4. **Comparative evaluation:** Compare all three configurations on consistent real-world and synthetic evaluation distributions.

## 3. Dataset Preparation

The previously established raw and processed synthetic datasets will be inspected before use.

The inspection will verify:

- Dataset schemas and available text and label fields.
- Existing preprocessing and normalization conventions.
- Binary label compatibility.
- Record counts and dataset provenance.
- Duplicate and near-duplicate relationships within and across synthetic datasets.
- Compatibility with the preprocessing and tokenization pipeline used in Notebook 06.

Previously processed datasets will be reused where compatible, avoiding unnecessary repeated preprocessing.

## 4. Evaluation Strategy

Two evaluation distributions will be considered:

- **Real-world evaluation:** The existing untouched group-aware real test partition containing real scam and real legitimate communications.
- **Synthetic evaluation:** Held-out synthetic scam examples paired with real legitimate test examples.

The synthetic evaluation will be explicitly identified as a mixed-distribution evaluation rather than an independent real-world benchmark.

## 5. Experimental Controls

- Preserve the existing real train, validation, and test partitions.
- Independently partition synthetic data before training.
- Prevent duplicate or near-duplicate records from crossing synthetic training, validation, and test partitions.
- Maintain consistent model architecture, tokenizer, preprocessing conventions, and evaluation metrics.
- Compare natural-size and size-controlled training configurations where appropriate.
- Keep all test examples outside training and model selection.

## 6. Evaluation Metrics

- Accuracy
- Scam-class precision
- Scam-class recall
- Scam-class F1-score
- Macro F1-score
- Precision-Recall AUC
- Confusion matrix

## 7. Research Questions

1. Can the existing real-only model recognize previously unseen synthetic scam narratives?
2. Does synthetic scam training improve detection of real financial scams?
3. Does combining real and synthetic scam data improve binary classification?
4. Are improvements attributable to synthetic data augmentation or simply increased training volume?

## 8. Interpretation Principles

Synthetic evaluation performance will not be treated as evidence of real-world generalization by itself.

Results will be interpreted according to dataset composition, provenance, and evaluation distribution. All findings, including negative or inconclusive results, will be reported transparently.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Verify the runtime environment, GPU, dataset files, and saved DistilBERT model

import os
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path("/content/financial-scam-analysis")
DRIVE_ROOT = Path("/content/drive/MyDrive/financial-scam-analysis")

REAL_SPLITS_DIR = DRIVE_ROOT / "data" / "splits"
SYNTHETIC_DATA_PATH = DRIVE_ROOT / "data" / "raw" / "synthetic_financial_scams.csv"
SAVED_MODEL_DIR = DRIVE_ROOT / "models" / "distilbert_binary"

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nProject root exists:", PROJECT_ROOT.exists())
print("Drive root exists:", DRIVE_ROOT.exists())

print("\nReal dataset partitions:")
for filename in ["train.csv", "validation.csv", "test.csv"]:
    path = REAL_SPLITS_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nSynthetic dataset:")
print(SYNTHETIC_DATA_PATH)
print("Status:", "FOUND" if SYNTHETIC_DATA_PATH.exists() else "MISSING")

print("\nSaved DistilBERT model:")
print(SAVED_MODEL_DIR)
print("Directory exists:", SAVED_MODEL_DIR.exists())

if SAVED_MODEL_DIR.exists():
    required_files = [
        "config.json",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json"
    ]

    for filename in required_files:
        path = SAVED_MODEL_DIR / filename
        print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4

Project root exists: False
Drive root exists: True

Real dataset partitions:
train.csv: FOUND
validation.csv: FOUND
test.csv: FOUND

Synthetic dataset:
/content/drive/MyDrive/financial-scam-analysis/data/raw/synthetic_financial_scams.csv
Status: MISSING

Saved DistilBERT model:
/content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
Directory exists: True
config.json: FOUND
model.safetensors: FOUND
tokenizer.json: FOUND
tokenizer_config.json: FOUND


In [3]:
# Locate the synthetic financial scam dataset within the project directory

from pathlib import Path

DATA_ROOT = DRIVE_ROOT / "data"

print("Available data directories:\n")

for path in sorted(DATA_ROOT.rglob("*")):
    if path.is_file() and (
        "synthetic" in path.name.lower()
        or "scam" in path.name.lower()
    ):
        print(path)

Available data directories:

/content/drive/MyDrive/financial-scam-analysis/data/processed/llm_authored_financial_scams_processed.csv
/content/drive/MyDrive/financial-scam-analysis/data/processed/real_fin_scam_processed.csv
/content/drive/MyDrive/financial-scam-analysis/data/processed/synthetic_financial_scams_v3_1_processed.csv
/content/drive/MyDrive/financial-scam-analysis/data/raw/llm_authored_financial_scams.csv
/content/drive/MyDrive/financial-scam-analysis/data/raw/real_fin_scam.csv
/content/drive/MyDrive/financial-scam-analysis/data/raw/synthetic_financial_scams_v3_1.csv


In [4]:
# Inspect raw and processed synthetic datasets and compare them with the established real scam dataset

import pandas as pd

DATA_PATHS = {
    "Real Scam (Processed)": DRIVE_ROOT / "data/processed/real_fin_scam_processed.csv",
    "V3.1 Synthetic (Raw)": DRIVE_ROOT / "data/raw/synthetic_financial_scams_v3_1.csv",
    "V3.1 Synthetic (Processed)": DRIVE_ROOT / "data/processed/synthetic_financial_scams_v3_1_processed.csv",
    "LLM-authored (Raw)": DRIVE_ROOT / "data/raw/llm_authored_financial_scams.csv",
    "LLM-authored (Processed)": DRIVE_ROOT / "data/processed/llm_authored_financial_scams_processed.csv",
}

datasets = {}

for name, path in DATA_PATHS.items():
    if not path.exists():
        print(f"\nMISSING: {name} -> {path}")
        continue

    df = pd.read_csv(path, low_memory=False)
    datasets[name] = df

    print(f"\n{'=' * 80}")
    print(name)
    print("=" * 80)
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes.to_string())

    for column in ["label", "scam_subtype", "fraudulent_intent", "social_engineering_tactics", "is_synthetic"]:
        if column in df.columns:
            print(f"\n{column} distribution:")
            print(df[column].value_counts(dropna=False).head(15).to_string())

    text_columns = [
        col for col in ["original_text", "normalized_text", "text", "message", "email_body"]
        if col in df.columns
    ]
    print("\nAvailable text columns:", text_columns)

    for column in text_columns:
        print(f"\nSample from {column}:")
        sample = df[column].dropna()
        if not sample.empty:
            print(str(sample.iloc[0])[:500])


Real Scam (Processed)
Shape: (13861, 30)

Columns:
['record_id', 'original_text', 'normalized_text', 'label', 'channel', 'original_channel', 'is_email_gateway', 'has_gateway_headers', 'source', 'source_url', 'source_license', 'source_record_id', 'is_synthetic', 'scam_subtype', 'fraudulent_intent', 'social_engineering_tactics', 'legitimate_category', 'email_subject', 'email_body', 'duplicate_cluster_id', 'char_count_original', 'word_count_original', 'char_count_normalized', 'word_count_normalized', 'placeholder_count_original', 'placeholder_count_normalized', 'generation_scenario', 'risk_or_safety_note', 'language', 'communication_type']

Data types:
record_id                        object
original_text                    object
normalized_text                  object
label                             int64
channel                          object
original_channel                 object
is_email_gateway                   bool
has_gateway_headers                bool
source               

In [5]:
# Audit existing synthetic partitions, duplicate clusters, and label consistency

for name in ["V3.1 Synthetic (Processed)", "LLM-authored (Processed)"]:
    df = datasets[name]

    print(f"\n{'=' * 75}")
    print(name)
    print("=" * 75)

    print("\nLabel values:")
    print(df["label"].value_counts(dropna=False).to_string())

    if "split_partition" in df.columns:
        print("\nExisting split_partition distribution:")
        print(df["split_partition"].value_counts(dropna=False).to_string())
    else:
        print("\nNo split_partition column found.")

    if "duplicate_cluster_id" in df.columns:
        cluster = df["duplicate_cluster_id"]
        print("\nDuplicate cluster audit:")
        print("Missing cluster IDs:", cluster.isna().sum())
        print("Unique non-null clusters:", cluster.nunique())
        print("Records in multi-record clusters:",
              cluster.value_counts().loc[lambda x: x > 1].sum()
              if cluster.notna().any() else 0)

    if "generation_scenario" in df.columns:
        print("\nGeneration scenario count:", df["generation_scenario"].nunique(dropna=True))

    if "scenario_family" in df.columns:
        print("Scenario family count:", df["scenario_family"].nunique(dropna=True))

    print("\nNormalized text missing:", df["normalized_text"].isna().sum())
    print("Normalized text empty:",
          df["normalized_text"].fillna("").str.strip().eq("").sum())

    print("Duplicate normalized texts:", df["normalized_text"].duplicated().sum())


V3.1 Synthetic (Processed)

Label values:
label
1    7800

Existing split_partition distribution:
split_partition
train    6240
val       780
test      780

Duplicate cluster audit:
Missing cluster IDs: 7627
Unique non-null clusters: 85
Records in multi-record clusters: 173

Generation scenario count: 104
Scenario family count: 104

Normalized text missing: 0
Normalized text empty: 0
Duplicate normalized texts: 43

LLM-authored (Processed)

Label values:
label
phishing    2730

Existing split_partition distribution:
split_partition
train    2184
val       273
test      273

Duplicate cluster audit:
Missing cluster IDs: 2730
Unique non-null clusters: 0
Records in multi-record clusters: 0
Scenario family count: 130

Normalized text missing: 0
Normalized text empty: 0
Duplicate normalized texts: 0


In [6]:
# Check whether duplicate records, duplicate clusters, or generation scenarios cross synthetic partitions

def audit_cross_partition_leakage(df, dataset_name):
    print(f"\n{'=' * 75}")
    print(dataset_name)
    print("=" * 75)

    for column in ["normalized_text", "duplicate_cluster_id", "generation_scenario", "scenario_family"]:
        if column not in df.columns:
            continue

        valid = df.dropna(subset=[column]).copy()

        if column == "normalized_text":
            valid = valid[valid[column].astype(str).str.strip().ne("")]

        group_partitions = valid.groupby(column)["split_partition"].nunique()
        crossing_groups = group_partitions[group_partitions > 1]

        print(f"\n{column}:")
        print("Unique groups:", valid[column].nunique())
        print("Groups appearing in multiple partitions:", len(crossing_groups))

        if len(crossing_groups):
            affected = valid[valid[column].isin(crossing_groups.index)]
            print("Affected records:", len(affected))
            print("Partition distribution of affected records:")
            print(affected["split_partition"].value_counts().to_string())
        else:
            print("No cross-partition overlap detected.")

for name in ["V3.1 Synthetic (Processed)", "LLM-authored (Processed)"]:
    audit_cross_partition_leakage(datasets[name], name)


V3.1 Synthetic (Processed)

normalized_text:
Unique groups: 7757
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

duplicate_cluster_id:
Unique groups: 85
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

generation_scenario:
Unique groups: 104
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

scenario_family:
Unique groups: 104
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

LLM-authored (Processed)

normalized_text:
Unique groups: 2730
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

duplicate_cluster_id:
Unique groups: 0
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.

scenario_family:
Unique groups: 130
Groups appearing in multiple partitions: 0
No cross-partition overlap detected.


In [7]:
# Load the untouched real test partition and verify its legitimate examples

REAL_TEST_PATH = REAL_SPLITS_DIR / "test.csv"

real_test_df = pd.read_csv(REAL_TEST_PATH, low_memory=False)

print("Real test shape:", real_test_df.shape)
print("\nLabel distribution:")
print(real_test_df["label"].value_counts(dropna=False).sort_index())

print("\nSynthetic flag distribution:")
if "is_synthetic" in real_test_df.columns:
    print(real_test_df["is_synthetic"].value_counts(dropna=False))

print("\nRequired columns:")
required_columns = ["normalized_text", "label"]
for column in required_columns:
    print(f"{column}: {'FOUND' if column in real_test_df.columns else 'MISSING'}")

real_legit_test = real_test_df.loc[
    real_test_df["label"].eq(0)
].copy()

print("\nLegitimate test records:", len(real_legit_test))
print("Missing normalized text:", real_legit_test["normalized_text"].isna().sum())
print(
    "Empty normalized text:",
    real_legit_test["normalized_text"].fillna("").str.strip().eq("").sum()
)

print("\nSample legitimate message:")
print(real_legit_test["normalized_text"].iloc[0][:500])

Real test shape: (7618, 32)

Label distribution:
label
0    6233
1    1385
Name: count, dtype: int64

Synthetic flag distribution:
is_synthetic
False    7618
Name: count, dtype: int64

Required columns:
normalized_text: FOUND
label: FOUND

Legitimate test records: 6233
Missing normalized text: 0
Empty normalized text: 0

Sample legitimate message:
Your OTP is 789654, keep it confidential to secure your account.


In [8]:
# Construct separate synthetic evaluation datasets using the existing test partitions

v31_df = datasets["V3.1 Synthetic (Processed)"]
llm_df = datasets["LLM-authored (Processed)"]

v31_scam_test = v31_df.loc[
    v31_df["split_partition"].eq("test"),
    ["normalized_text"]
].copy()

v31_scam_test["label"] = 1
v31_scam_test["source"] = "V3.1 Synthetic"

llm_scam_test = llm_df.loc[
    llm_df["split_partition"].eq("test"),
    ["normalized_text"]
].copy()

llm_scam_test["label"] = 1
llm_scam_test["source"] = "LLM-authored"

legit_eval = real_legit_test[["normalized_text"]].copy()
legit_eval["label"] = 0
legit_eval["source"] = "Real Legitimate"

v31_eval_df = pd.concat(
    [v31_scam_test, legit_eval],
    ignore_index=True
)

llm_eval_df = pd.concat(
    [llm_scam_test, legit_eval],
    ignore_index=True
)

print("V3.1 evaluation set:", v31_eval_df.shape)
print(v31_eval_df["label"].value_counts().sort_index())
print("\nV3.1 source distribution:")
print(v31_eval_df["source"].value_counts())

print("\nLLM-authored evaluation set:", llm_eval_df.shape)
print(llm_eval_df["label"].value_counts().sort_index())
print("\nLLM-authored source distribution:")
print(llm_eval_df["source"].value_counts())

print("\nData integrity:")
for name, df in [
    ("V3.1", v31_eval_df),
    ("LLM-authored", llm_eval_df)
]:
    print(
        f"{name}: missing={df['normalized_text'].isna().sum()}, "
        f"empty={df['normalized_text'].fillna('').str.strip().eq('').sum()}"
    )

print(
    "\nIdentical legitimate evaluation examples:",
    v31_eval_df.loc[v31_eval_df["label"].eq(0), "normalized_text"].reset_index(drop=True).equals(
        llm_eval_df.loc[llm_eval_df["label"].eq(0), "normalized_text"].reset_index(drop=True)
    )
)

V3.1 evaluation set: (7013, 3)
label
0    6233
1     780
Name: count, dtype: int64

V3.1 source distribution:
source
Real Legitimate    6233
V3.1 Synthetic      780
Name: count, dtype: int64

LLM-authored evaluation set: (6506, 3)
label
0    6233
1     273
Name: count, dtype: int64

LLM-authored source distribution:
source
Real Legitimate    6233
LLM-authored        273
Name: count, dtype: int64

Data integrity:
V3.1: missing=0, empty=0
LLM-authored: missing=0, empty=0

Identical legitimate evaluation examples: True


In [9]:
# Load the previously trained DistilBERT binary classifier

from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = str(SAVED_MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path
).to(device)

model.eval()

print("Model loaded successfully.")
print("Model:", model.config._name_or_path)
print("Device:", device)
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)
print("Maximum sequence length:", model.config.max_position_embeddings)
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Model in evaluation mode:", not model.training)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.
Model: /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
Device: cuda
Number of labels: 2
Label mapping: {0: 'LEGITIMATE', 1: 'SCAM'}
Maximum sequence length: 512
Model parameters: 66955010
Model in evaluation mode: True


In [10]:
# Run zero-shot inference on V3.1 and LLM-authored synthetic evaluation sets

import numpy as np
from tqdm.auto import tqdm

def run_inference(model, tokenizer, dataframe, batch_size=32, max_length=512):
    texts = dataframe["normalized_text"].tolist()
    all_probabilities = []

    model.eval()

    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[start:start + batch_size]

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device)

            outputs = model(**encoded)
            probabilities = torch.softmax(outputs.logits, dim=-1)

            all_probabilities.append(probabilities.cpu().numpy())

    probabilities = np.concatenate(all_probabilities, axis=0)
    predictions = probabilities.argmax(axis=1)

    results = dataframe.copy()
    results["predicted_label"] = predictions
    results["legitimate_probability"] = probabilities[:, 0]
    results["scam_probability"] = probabilities[:, 1]

    return results


v31_predictions = run_inference(
    model, tokenizer, v31_eval_df
)

llm_predictions = run_inference(
    model, tokenizer, llm_eval_df
)

print("Inference completed.")
print("V3.1 predictions:", v31_predictions.shape)
print("LLM-authored predictions:", llm_predictions.shape)

print("\nV3.1 prediction distribution:")
print(v31_predictions["predicted_label"].value_counts().sort_index())

print("\nLLM-authored prediction distribution:")
print(llm_predictions["predicted_label"].value_counts().sort_index())

  0%|          | 0/220 [00:00<?, ?it/s]

  0%|          | 0/204 [00:00<?, ?it/s]

Inference completed.
V3.1 predictions: (7013, 6)
LLM-authored predictions: (6506, 6)

V3.1 prediction distribution:
predicted_label
0    6227
1     786
Name: count, dtype: int64

LLM-authored prediction distribution:
predicted_label
0    6230
1     276
Name: count, dtype: int64


In [11]:
# Evaluate the existing DistilBERT model on both synthetic scam distributions

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    average_precision_score
)

def evaluate_predictions(results, dataset_name):
    y_true = results["label"].astype(int)
    y_pred = results["predicted_label"].astype(int)
    scam_probabilities = results["scam_probability"]

    print(f"\n{'=' * 75}")
    print(dataset_name)
    print("=" * 75)

    print("Total records:", len(results))
    print("Actual scams:", int(y_true.sum()))
    print("Actual legitimate:", int((y_true == 0).sum()))
    print("Predicted scams:", int(y_pred.sum()))

    print("\nClassification metrics:")
    print(f"Accuracy:       {accuracy_score(y_true, y_pred):.6f}")
    print(f"Scam Precision: {precision_score(y_true, y_pred, zero_division=0):.6f}")
    print(f"Scam Recall:    {recall_score(y_true, y_pred, zero_division=0):.6f}")
    print(f"Scam F1:        {f1_score(y_true, y_pred, zero_division=0):.6f}")
    print(f"Macro F1:       {f1_score(y_true, y_pred, average='macro', zero_division=0):.6f}")
    print(f"PR-AUC:         {average_precision_score(y_true, scam_probabilities):.6f}")

    print("\nConfusion Matrix:")
    print("Rows = Actual, Columns = Predicted")
    print("Labels = [LEGITIMATE, SCAM]")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["LEGITIMATE", "SCAM"],
        digits=4,
        zero_division=0
    ))


evaluate_predictions(v31_predictions, "V3.1 Synthetic Evaluation")
evaluate_predictions(llm_predictions, "LLM-authored Synthetic Evaluation")


V3.1 Synthetic Evaluation
Total records: 7013
Actual scams: 780
Actual legitimate: 6233
Predicted scams: 786

Classification metrics:
Accuracy:       0.998859
Scam Precision: 0.991094
Scam Recall:    0.998718
Scam F1:        0.994891
Macro F1:       0.997125
PR-AUC:         0.999443

Confusion Matrix:
Rows = Actual, Columns = Predicted
Labels = [LEGITIMATE, SCAM]
[[6226    7]
 [   1  779]]

Classification Report:
              precision    recall  f1-score   support

  LEGITIMATE     0.9998    0.9989    0.9994      6233
        SCAM     0.9911    0.9987    0.9949       780

    accuracy                         0.9989      7013
   macro avg     0.9955    0.9988    0.9971      7013
weighted avg     0.9989    0.9989    0.9989      7013


LLM-authored Synthetic Evaluation
Total records: 6506
Actual scams: 273
Actual legitimate: 6233
Predicted scams: 276

Classification metrics:
Accuracy:       0.998309
Scam Precision: 0.974638
Scam Recall:    0.985348
Scam F1:        0.979964
Macro F1:   

In [12]:
# Inspect synthetic scam messages incorrectly classified as legitimate

false_negatives = pd.concat(
    [
        v31_predictions.loc[
            (v31_predictions["label"] == 1) &
            (v31_predictions["predicted_label"] == 0)
        ],
        llm_predictions.loc[
            (llm_predictions["label"] == 1) &
            (llm_predictions["predicted_label"] == 0)
        ]
    ],
    ignore_index=True
).copy()

false_negatives["token_count"] = false_negatives["normalized_text"].apply(
    lambda text: len(
        tokenizer(
            str(text),
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )
)

false_negatives = false_negatives.sort_values(
    "scam_probability"
).reset_index(drop=True)

print("Total false negatives:", len(false_negatives))

for index, row in false_negatives.iterrows():
    print(f"\n{'=' * 90}")
    print(f"False Negative {index + 1}")
    print("=" * 90)
    print("Source:", row["source"])
    print("Actual label:", row["label"])
    print("Predicted label:", row["predicted_label"])
    print(f"Scam probability: {row['scam_probability']:.6f}")
    print(f"Legitimate probability: {row['legitimate_probability']:.6f}")
    print("Token count:", row["token_count"])
    print("Exceeds 512 tokens:", row["token_count"] > 512)
    print("\nMessage:")
    print(row["normalized_text"])

Total false negatives: 5

False Negative 1
Source: LLM-authored
Actual label: 1
Predicted label: 0
Scam probability: 0.001919
Legitimate probability: 0.998081
Token count: 77
Exceeds 512 tokens: False

Message:
hey buddy you seeing the flash floods down in the valley right now?? whole towns are underwater and thousands of families are stranded with zero dry food or drinking water. a bunch of us set up an emergency supplies pool through Solaria Aid. even pitching in 15 or 20 bucks gets emergency ration packs delivered by boat tonight. tap here to chip in: [URL]

False Negative 2
Source: V3.1 Synthetic
Actual label: 1
Predicted label: 0
Scam probability: 0.001932
Legitimate probability: 0.998068
Token count: 141
Exceeds 512 tokens: False

Message:
[11:20 AM] Talent_Coordinator_Alex: E-Commerce Role: Part-time product cataloging assistant position with flexible hours and weekly payments. [11:22 AM] Applicant: Is prior experience required, or is training provided? [11:23 AM] Talent_Coordin

In [13]:
# Recover original research annotations for the five false-negative examples

metadata_columns = [
    "normalized_text",
    "scam_subtype",
    "fraudulent_intent",
    "social_engineering_tactics",
    "primary_intent_evidence",
    "tactics_evidence",
    "narrative_evidence",
    "scenario_family",
    "generation_scenario",
    "generation_template_id",
    "phishing_type",
    "writing_style",
    "urgency_rating"
]

metadata_frames = []

for dataset_name, source_name in [
    ("V3.1 Synthetic (Processed)", "V3.1 Synthetic"),
    ("LLM-authored (Processed)", "LLM-authored")
]:
    source_df = datasets[dataset_name]

    available_columns = [
        column for column in metadata_columns
        if column in source_df.columns
    ]

    source_metadata = source_df[available_columns].copy()
    source_metadata = source_metadata.drop_duplicates(
        subset=["normalized_text"]
    )

    source_metadata["annotation_source"] = source_name
    metadata_frames.append(source_metadata)

synthetic_metadata = pd.concat(
    metadata_frames,
    ignore_index=True
)

false_negatives_annotated = false_negatives.merge(
    synthetic_metadata,
    on="normalized_text",
    how="left",
    validate="many_to_one"
)

print("False negatives:", len(false_negatives_annotated))
print(
    "Successfully matched:",
    false_negatives_annotated["annotation_source"].notna().sum()
)

display_columns = [
    "source",
    "scam_probability",
    "scam_subtype",
    "fraudulent_intent",
    "social_engineering_tactics",
    "scenario_family",
    "generation_scenario",
    "writing_style",
    "urgency_rating"
]

display(
    false_negatives_annotated[
        [column for column in display_columns
         if column in false_negatives_annotated.columns]
    ].to_string(index=False)
)

False negatives: 5
Successfully matched: 5


'        source  scam_probability            scam_subtype             fraudulent_intent                                  social_engineering_tactics                     scenario_family       generation_scenario           writing_style urgency_rating\n  LLM-authored          0.001919        charity_donation    induce_payment_or_transfer ["emotional_manipulation", "urgency", "trust_exploitation"] flood_catastrophe_emergency_rations                       NaN         casual_informal         medium\nV3.1 Synthetic          0.001932     job_task_commission gain_account_or_device_access                             ["urgency", "false_legitimacy"]           product_listing_assistant product_listing_assistant                     NaN            NaN\n  LLM-authored          0.015304        charity_donation    induce_payment_or_transfer ["emotional_manipulation", "urgency", "trust_exploitation"] flood_catastrophe_emergency_rations                       NaN         casual_informal           high\n  L

In [14]:
# Examine the complete annotation evidence for all five false negatives

evidence_columns = [
    "source",
    "scam_subtype",
    "fraudulent_intent",
    "social_engineering_tactics",
    "primary_intent_evidence",
    "tactics_evidence",
    "narrative_evidence",
    "scenario_family",
    "generation_scenario",
    "writing_style",
    "urgency_rating"
]

for index, row in false_negatives_annotated.iterrows():
    print(f"\n{'=' * 90}")
    print(f"FALSE NEGATIVE {index + 1}")
    print("=" * 90)

    print("Scam probability:", round(row["scam_probability"], 6))

    for column in evidence_columns:
        if column in row.index:
            value = row[column]
            if pd.notna(value):
                print(f"\n{column}:")
                print(value)


FALSE NEGATIVE 1
Scam probability: 0.001919

source:
LLM-authored

scam_subtype:
charity_donation

fraudulent_intent:
induce_payment_or_transfer

social_engineering_tactics:
["emotional_manipulation", "urgency", "trust_exploitation"]

primary_intent_evidence:
tap here to chip in: https://relief.solaria-aid.example/valley-pool

tactics_evidence:
[{"tactic": "emotional_manipulation", "span_text": "thousands of families are stranded with zero dry food or drinking water", "start": 100, "end": 171}, {"tactic": "urgency", "span_text": "gets emergency ration packs delivered by boat tonight", "start": 274, "end": 327}, {"tactic": "trust_exploitation", "span_text": "hey buddy you seeing the flash floods down in the valley right now??", "start": 0, "end": 68}]

scenario_family:
flood_catastrophe_emergency_rations

writing_style:
casual_informal

urgency_rating:
medium

FALSE NEGATIVE 2
Scam probability: 0.001932

source:
V3.1 Synthetic

scam_subtype:
job_task_commission

fraudulent_intent:
gain

In [15]:
# Analyze synthetic scam detection performance by subtype and scenario family

subtype_results = []
scenario_results = []

for dataset_name, predictions in [
    ("V3.1 Synthetic", v31_predictions),
    ("LLM-authored", llm_predictions)
]:
    original_df = datasets[
        "V3.1 Synthetic (Processed)"
        if dataset_name == "V3.1 Synthetic"
        else "LLM-authored (Processed)"
    ]

    test_metadata = original_df.loc[
        original_df["split_partition"].eq("test")
    ].copy()

    metadata_columns = [
        "normalized_text",
        "scam_subtype",
        "fraudulent_intent",
        "scenario_family"
    ]

    test_metadata = test_metadata[metadata_columns].copy()

    scam_results = predictions.loc[
        predictions["label"].eq(1)
    ].copy()

    # Preserve repeated messages by assigning an occurrence index
    scam_results["text_occurrence"] = (
        scam_results.groupby("normalized_text").cumcount()
    )

    test_metadata["text_occurrence"] = (
        test_metadata.groupby("normalized_text").cumcount()
    )

    scam_results = scam_results.merge(
        test_metadata,
        on=["normalized_text", "text_occurrence"],
        how="left",
        validate="one_to_one"
    )

    assert len(scam_results) == len(predictions[predictions["label"].eq(1)])
    assert scam_results["scam_subtype"].notna().all()

    for subtype, group in scam_results.groupby("scam_subtype"):
        subtype_results.append({
            "dataset": dataset_name,
            "subtype": subtype,
            "support": len(group),
            "correct": int(group["predicted_label"].eq(1).sum()),
            "missed": int(group["predicted_label"].eq(0).sum()),
            "recall": group["predicted_label"].eq(1).mean()
        })

    for scenario, group in scam_results.groupby("scenario_family"):
        scenario_results.append({
            "dataset": dataset_name,
            "scenario_family": scenario,
            "support": len(group),
            "correct": int(group["predicted_label"].eq(1).sum()),
            "missed": int(group["predicted_label"].eq(0).sum()),
            "recall": group["predicted_label"].eq(1).mean()
        })

subtype_performance = pd.DataFrame(subtype_results)
scenario_performance = pd.DataFrame(scenario_results)

print("SUBTYPE-WISE PERFORMANCE")
display(
    subtype_performance.sort_values(
        ["dataset", "recall", "support"]
    ).reset_index(drop=True)
)

print("\nSCENARIO FAMILIES WITH MISSED SCAMS")
display(
    scenario_performance.loc[
        scenario_performance["missed"] > 0
    ].sort_values(
        ["dataset", "missed"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

SUBTYPE-WISE PERFORMANCE


,dataset,subtype,support,correct,missed,recall
0,LLM-authored,charity_donation,21,19,2,0.904762
1,LLM-authored,invoice_payment_request,21,19,2,0.904762
2,LLM-authored,account_verification,21,21,0,1.000000
3,LLM-authored,card_transaction_alert,21,21,0,1.000000
4,LLM-authored,family_emergency_request,21,21,0,1.000000
5,LLM-authored,investment_crypto,21,21,0,1.000000
6,LLM-authored,job_task_commission,21,21,0,1.000000
7,LLM-authored,loan_offer,21,21,0,1.000000
8,LLM-authored,prize_reward,21,21,0,1.000000
9,LLM-authored,recovery_support,21,21,0,1.000000



SCENARIO FAMILIES WITH MISSED SCAMS


,dataset,scenario_family,support,correct,missed,recall
0,LLM-authored,commercial_printing_packaging_order,21,19,2,0.904762
1,LLM-authored,flood_catastrophe_emergency_rations,21,19,2,0.904762
2,V3.1 Synthetic,product_listing_assistant,60,59,1,0.983333


In [16]:
# Compare scam probability distributions across synthetic scam subtypes

confidence_results = []

for dataset_name, predictions in [
    ("V3.1 Synthetic", v31_predictions),
    ("LLM-authored", llm_predictions)
]:
    original_df = datasets[
        "V3.1 Synthetic (Processed)"
        if dataset_name == "V3.1 Synthetic"
        else "LLM-authored (Processed)"
    ]

    test_metadata = original_df.loc[
        original_df["split_partition"].eq("test"),
        ["normalized_text", "scam_subtype"]
    ].copy()

    test_metadata["text_occurrence"] = (
        test_metadata.groupby("normalized_text").cumcount()
    )

    scam_results = predictions.loc[
        predictions["label"].eq(1)
    ].copy()

    scam_results["text_occurrence"] = (
        scam_results.groupby("normalized_text").cumcount()
    )

    scam_results = scam_results.merge(
        test_metadata,
        on=["normalized_text", "text_occurrence"],
        how="left",
        validate="one_to_one"
    )

    for subtype, group in scam_results.groupby("scam_subtype"):
        confidence_results.append({
            "dataset": dataset_name,
            "subtype": subtype,
            "support": len(group),
            "mean_scam_probability": group["scam_probability"].mean(),
            "median_scam_probability": group["scam_probability"].median(),
            "minimum_scam_probability": group["scam_probability"].min(),
            "mean_probability_correct": group.loc[
                group["predicted_label"].eq(1),
                "scam_probability"
            ].mean(),
            "mean_probability_missed": group.loc[
                group["predicted_label"].eq(0),
                "scam_probability"
            ].mean()
        })

confidence_by_subtype = pd.DataFrame(confidence_results)

display(
    confidence_by_subtype.sort_values(
        ["dataset", "mean_scam_probability"]
    ).reset_index(drop=True)
)

,dataset,subtype,support,mean_scam_probability,median_scam_probability,minimum_scam_probability,mean_probability_correct,mean_probability_missed
0,LLM-authored,charity_donation,21,0.904651,0.999975,0.001919,0.998971,0.008611
1,LLM-authored,invoice_payment_request,21,0.905484,0.999968,0.046128,0.985788,0.142588
2,LLM-authored,investment_crypto,21,0.977251,0.999985,0.523207,0.977251,NaN
3,LLM-authored,job_task_commission,21,0.978381,0.999972,0.555419,0.978381,NaN
4,LLM-authored,family_emergency_request,21,0.999756,0.999979,0.995847,0.999756,NaN
5,LLM-authored,romance_financial_manipulation,21,0.999911,0.999984,0.998590,0.999911,NaN
6,LLM-authored,account_verification,21,0.999977,0.999985,0.999928,0.999977,NaN
7,LLM-authored,refund_chargeback,21,0.999981,0.999985,0.999897,0.999981,NaN
8,LLM-authored,prize_reward,21,0.999981,0.999985,0.999942,0.999981,NaN
9,LLM-authored,loan_offer,21,0.999983,0.999986,0.999960,0.999983,NaN


In [17]:
# Load the real training and validation partitions for augmentation experiments

real_train_df = pd.read_csv(
    DRIVE_ROOT / "data/splits/train.csv"
)

real_validation_df = pd.read_csv(
    DRIVE_ROOT / "data/splits/validation.csv"
)

print("Real training partition:", real_train_df.shape)
print("Real validation partition:", real_validation_df.shape)

print("\nTraining label distribution:")
display(real_train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
display(real_validation_df["label"].value_counts().sort_index())

print("\nSynthetic records in real training partition:",
      real_train_df["is_synthetic"].sum())

print("Synthetic records in real validation partition:",
      real_validation_df["is_synthetic"].sum())

/tmp/ipykernel_408/1170067203.py:3: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  real_train_df = pd.read_csv(


Real training partition: (60957, 32)
Real validation partition: (7618, 32)

Training label distribution:


,count
label,
0,49866
1,11091



Validation label distribution:


,count
label,
0,6233
1,1385



Synthetic records in real training partition: 0
Synthetic records in real validation partition: 0


In [18]:
# Verify synthetic labels and partition distributions before training-data construction

for dataset_name in [
    "V3.1 Synthetic (Processed)",
    "LLM-authored (Processed)"
]:
    df = datasets[dataset_name]

    print(f"\n{'=' * 65}")
    print(dataset_name)
    print("=" * 65)

    print("\nLabel values:")
    display(df["label"].value_counts(dropna=False))

    print("\nPartition distribution:")
    display(
        df["split_partition"]
        .value_counts(dropna=False)
        .sort_index()
    )

    print("\nSynthetic flag:")
    display(df["is_synthetic"].value_counts(dropna=False))

    print("\nMissing normalized text:",
          df["normalized_text"].isna().sum())

    print("Empty normalized text:",
          df["normalized_text"].fillna("").str.strip().eq("").sum())


V3.1 Synthetic (Processed)

Label values:


,count
label,
1,7800



Partition distribution:


,count
split_partition,
test,780
train,6240
val,780



Synthetic flag:


,count
is_synthetic,
True,7800


... (3 outputs left out)

,count
split_partition,
test,273
train,2184
val,273



Synthetic flag:


,count
is_synthetic,
True,2730



Missing normalized text: 0
Empty normalized text: 0


In [19]:
# Harmonize the two synthetic datasets into a common binary-label format

synthetic_train_parts = []
synthetic_validation_parts = []

for dataset_name, source_name in [
    ("V3.1 Synthetic (Processed)", "V3.1 Synthetic"),
    ("LLM-authored (Processed)", "LLM-authored")
]:
    df = datasets[dataset_name]

    for partition, destination in [
        ("train", synthetic_train_parts),
        ("val", synthetic_validation_parts)
    ]:
        subset = df.loc[
            df["split_partition"].eq(partition),
            ["normalized_text", "label", "scam_subtype", "channel"]
        ].copy()

        subset["label"] = 1
        subset["synthetic_source"] = source_name

        destination.append(subset)

synthetic_train_df = pd.concat(
    synthetic_train_parts,
    ignore_index=True
)

synthetic_validation_df = pd.concat(
    synthetic_validation_parts,
    ignore_index=True
)

print("Synthetic training:", synthetic_train_df.shape)
print("Synthetic validation:", synthetic_validation_df.shape)

print("\nTraining distribution by source:")
display(
    synthetic_train_df["synthetic_source"].value_counts()
)

print("\nValidation distribution by source:")
display(
    synthetic_validation_df["synthetic_source"].value_counts()
)

print("\nHarmonized labels:")
display(synthetic_train_df["label"].value_counts())

Synthetic training: (8424, 5)
Synthetic validation: (1053, 5)

Training distribution by source:


,count
synthetic_source,
V3.1 Synthetic,6240
LLM-authored,2184



Validation distribution by source:


,count
synthetic_source,
V3.1 Synthetic,780
LLM-authored,273



Harmonized labels:


,count
label,
1,8424


In [20]:
# Construct the synthetic-only and combined training and validation datasets

real_legit_train = real_train_df.loc[
    real_train_df["label"].eq(0),
    ["normalized_text", "label"]
].copy()

real_legit_validation = real_validation_df.loc[
    real_validation_df["label"].eq(0),
    ["normalized_text", "label"]
].copy()

experiment_b_train = pd.concat(
    [real_legit_train, synthetic_train_df[["normalized_text", "label"]]],
    ignore_index=True
)

experiment_b_validation = pd.concat(
    [real_legit_validation, synthetic_validation_df[["normalized_text", "label"]]],
    ignore_index=True
)

experiment_c_train = pd.concat(
    [
        real_train_df[["normalized_text", "label"]],
        synthetic_train_df[["normalized_text", "label"]]
    ],
    ignore_index=True
)

experiment_c_validation = pd.concat(
    [
        real_validation_df[["normalized_text", "label"]],
        synthetic_validation_df[["normalized_text", "label"]]
    ],
    ignore_index=True
)

for name, train_data, validation_data in [
    ("Experiment B — Synthetic + Real Legitimate",
     experiment_b_train, experiment_b_validation),
    ("Experiment C — Real + Synthetic",
     experiment_c_train, experiment_c_validation)
]:
    print(f"\n{'=' * 65}")
    print(name)
    print("=" * 65)

    print("Training:", train_data.shape)
    display(train_data["label"].value_counts().sort_index())

    print("Validation:", validation_data.shape)
    display(validation_data["label"].value_counts().sort_index())

    print("Training scam prevalence:",
          round(train_data["label"].mean() * 100, 2), "%")

    print("Validation scam prevalence:",
          round(validation_data["label"].mean() * 100, 2), "%")


Experiment B — Synthetic + Real Legitimate
Training: (58290, 2)


,count
label,
0,49866
1,8424


Validation: (7286, 2)


,count
label,
0,6233
1,1053


Training scam prevalence: 14.45 %
Validation scam prevalence: 14.45 %

Experiment C — Real + Synthetic
Training: (69381, 2)


,count
label,
0,49866
1,19515


Validation: (8671, 2)


,count
label,
0,6233
1,2438


Training scam prevalence: 28.13 %
Validation scam prevalence: 28.12 %


In [21]:
# Check exact normalized-text overlap between synthetic data and real evaluation partitions

real_test_texts = set(
    real_test_df["normalized_text"].dropna().str.strip()
)

real_train_texts = set(
    real_train_df["normalized_text"].dropna().str.strip()
)

real_validation_texts = set(
    real_validation_df["normalized_text"].dropna().str.strip()
)

for name, synthetic_df in [
    ("Synthetic training", synthetic_train_df),
    ("Synthetic validation", synthetic_validation_df)
]:
    synthetic_texts = set(
        synthetic_df["normalized_text"].dropna().str.strip()
    )

    print(f"\n{name}")
    print("-" * 45)
    print("Unique synthetic messages:", len(synthetic_texts))
    print("Overlap with real training:",
          len(synthetic_texts & real_train_texts))
    print("Overlap with real validation:",
          len(synthetic_texts & real_validation_texts))
    print("Overlap with real test:",
          len(synthetic_texts & real_test_texts))


Synthetic training
---------------------------------------------
Unique synthetic messages: 8387
Overlap with real training: 0
Overlap with real validation: 0
Overlap with real test: 0

Synthetic validation
---------------------------------------------
Unique synthetic messages: 1050
Overlap with real training: 0
Overlap with real validation: 0
Overlap with real test: 0


In [22]:
# Tokenize both augmentation experiments using the Notebook06 tokenizer and sequence length

from datasets import Dataset

MAX_LENGTH = 512

def tokenize_experiment(dataframe):
    dataset = Dataset.from_pandas(
        dataframe[["normalized_text", "label"]].reset_index(drop=True),
        preserve_index=False
    )

    def tokenize_batch(batch):
        return tokenizer(
            batch["normalized_text"],
            truncation=True,
            padding=False,
            max_length=MAX_LENGTH
        )

    tokenized = dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=["normalized_text"]
    )

    tokenized = tokenized.rename_column("label", "labels")
    return tokenized

experiment_b_train_tokenized = tokenize_experiment(experiment_b_train)
experiment_b_val_tokenized = tokenize_experiment(experiment_b_validation)

experiment_c_train_tokenized = tokenize_experiment(experiment_c_train)
experiment_c_val_tokenized = tokenize_experiment(experiment_c_validation)

print("Experiment B:")
print("Train:", len(experiment_b_train_tokenized))
print("Validation:", len(experiment_b_val_tokenized))

print("\nExperiment C:")
print("Train:", len(experiment_c_train_tokenized))
print("Validation:", len(experiment_c_val_tokenized))

print("\nTokenized example:")
print(experiment_b_train_tokenized[0].keys())
print("Token count:", len(experiment_b_train_tokenized[0]["input_ids"]))

Map:   0%|          | 0/58290 [00:00<?, ? examples/s]

Map:   0%|          | 0/7286 [00:00<?, ? examples/s]

Map:   0%|          | 0/69381 [00:00<?, ? examples/s]

Map:   0%|          | 0/8671 [00:00<?, ? examples/s]

Experiment B:
Train: 58290
Validation: 7286

Experiment C:
Train: 69381
Validation: 8671

Tokenized example:
dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])
Token count: 8


In [23]:
# Inspect the existing DistilBERT model configuration before reproducing training

print("Model:", model.config._name_or_path)
print("Architecture:", model.config.architectures)
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print("\nTokenizer:")
print("Maximum model length:", tokenizer.model_max_length)
print("Padding token:", tokenizer.pad_token)
print("Padding token ID:", tokenizer.pad_token_id)

print("\nTraining hardware:")
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Model: /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
Architecture: ['DistilBertForSequenceClassification']
Number of labels: 2
Label mapping: {0: 'LEGITIMATE', 1: 'SCAM'}

Tokenizer:
Maximum model length: 512
Padding token: [PAD]
Padding token ID: 0

Training hardware:
Device: cuda
CUDA available: True
GPU: Tesla T4


In [24]:
# Define the common training configuration to reproduce Notebook06

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

MODEL_NAME = "distilbert/distilbert-base-uncased"

TRAINING_CONFIG = {
    "num_train_epochs": 3,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "fp16": True,
    "max_length": 512,
    "seed": 42
}

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)


print("Shared training configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"{key}: {value}")

print("\nDynamic padding collator ready.")

Shared training configuration:
num_train_epochs: 3
per_device_train_batch_size: 16
per_device_eval_batch_size: 16
learning_rate: 2e-05
weight_decay: 0.01
fp16: True
max_length: 512
seed: 42

Dynamic padding collator ready.


In [26]:
# Define a reproducible DistilBERT training function for both augmentation experiments

import gc
import random
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    average_precision_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.softmax(
        torch.tensor(logits), dim=1
    ).numpy()

    predictions = np.argmax(probabilities, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    macro_f1 = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )[2]

    return {
        "accuracy": accuracy_score(labels, predictions),
        "scam_precision": precision,
        "scam_recall": recall,
        "scam_f1": f1,
        "macro_f1": macro_f1,
        "pr_auc": average_precision_score(
            labels, probabilities[:, 1]
        )
    }


def train_augmentation_model(
    experiment_name,
    train_dataset,
    validation_dataset
):
    gc.collect()
    torch.cuda.empty_cache()

    random.seed(TRAINING_CONFIG["seed"])
    np.random.seed(TRAINING_CONFIG["seed"])
    torch.manual_seed(TRAINING_CONFIG["seed"])
    torch.cuda.manual_seed_all(TRAINING_CONFIG["seed"])

    output_dir = (
        DRIVE_ROOT / "outputs" / f"distilbert_{experiment_name.lower()}"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    fresh_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={0: "LEGITIMATE", 1: "SCAM"},
        label2id={"LEGITIMATE": 0, "SCAM": 1}
    )

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG[
            "per_device_train_batch_size"
        ],
        per_device_eval_batch_size=TRAINING_CONFIG[
            "per_device_eval_batch_size"
        ],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        weight_decay=TRAINING_CONFIG["weight_decay"],
        fp16=TRAINING_CONFIG["fp16"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_macro_f1",
        greater_is_better=True,
        logging_strategy="epoch",
        seed=TRAINING_CONFIG["seed"],
        data_seed=TRAINING_CONFIG["seed"],
        report_to="none"
    )

    trainer = Trainer(
        model=fresh_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    print(f"\nStarting {experiment_name}")
    print("Training examples:", len(train_dataset))
    print("Validation examples:", len(validation_dataset))
    print("Output directory:", output_dir)

    train_result = trainer.train()

    print("\nTraining completed.")
    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    print("Best validation metric:", trainer.state.best_metric)

    return trainer, train_result, output_dir